In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

# Clinical Data
df = pd.read_csv("./X_train/clinical_train.csv")
df_eval = pd.read_csv("./X_test/clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("./X_train/molecular_train.csv")
maf_eval = pd.read_csv("./X_test/molecular_test.csv")

target_df = pd.read_csv("./target_train.csv")
#target_df_test = pd.read_csv("./target_test.csv")

# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


### Step 1: Data Preparation (clinical data only)

For survival analysis, we’ll format the dataset so that OS_YEARS represents the time variable and OS_STATUS represents the event indicator.

In [2]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Select features
features = ['BM_BLAST', 'HB', 'PLT']
target = ['OS_YEARS', 'OS_STATUS']

# Create the survival data format
X = df.loc[df['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

OS_STATUS    float64
OS_YEARS     float64
dtype: object


### Step 2: Splitting the Dataset
We’ll split the data into training and testing sets to evaluate the model’s performance.

In [3]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, y_train = X, y

In [4]:
# Survival-aware imputation for missing values
imputer = SimpleImputer(strategy="median")
X_train[['BM_BLAST', 'HB', 'PLT']] = imputer.fit_transform(X_train[['BM_BLAST', 'HB', 'PLT']])
X_test[['BM_BLAST', 'HB', 'PLT']] = imputer.transform(X_test[['BM_BLAST', 'HB', 'PLT']])

In [5]:
# Import additional libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
import re


# ==== 1. ENHANCED FEATURE ENGINEERING ====

# 1.1 Extract all available clinical features
clinical_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT', 'CENTER']
X_clinical = df.loc[df['ID'].isin(target_df['ID']), clinical_features + ['ID']]

# 1.2 Process cytogenetic data to extract actionable features
# Define cytogenetic feature columns globally
cyto_features = {
    'normal_karyotype': [],
    'complex_karyotype': [],
    'monosomy_5': [],
    'monosomy_7': [],
    'trisomy_8': [],
    'del_5q': [],
    'del_7q': [],
    'inv_16': [],
    't_8_21': [],
    't_15_17': []
}

def extract_cytogenetic_features(df):
    df_cyto = df.copy()
    
    # Create fresh empty lists for each feature
    feature_values = {feature: [] for feature in cyto_features}
    
    for idx, row in df.iterrows():
        cyto = str(row['CYTOGENETICS']).lower() if pd.notna(row['CYTOGENETICS']) else ''
        
        # Normal karyotype
        feature_values['normal_karyotype'].append(1 if re.search(r'46,x[xy]', cyto) is not None and len(cyto) < 10 else 0)
        
        # Complex karyotype (3 or more abnormalities)
        feature_values['complex_karyotype'].append(1 if cyto.count(',') > 3 else 0)
        
        # Specific abnormalities
        feature_values['monosomy_5'].append(1 if '-5' in cyto or 'monosomy 5' in cyto else 0)
        feature_values['monosomy_7'].append(1 if '-7' in cyto or 'monosomy 7' in cyto else 0)
        feature_values['trisomy_8'].append(1 if '+8' in cyto or 'trisomy 8' in cyto else 0)
        feature_values['del_5q'].append(1 if 'del(5q)' in cyto or '5q-' in cyto else 0)
        feature_values['del_7q'].append(1 if 'del(7q)' in cyto or '7q-' in cyto else 0)
        feature_values['inv_16'].append(1 if 'inv(16)' in cyto else 0)
        feature_values['t_8_21'].append(1 if 't(8;21)' in cyto else 0)
        feature_values['t_15_17'].append(1 if 't(15;17)' in cyto else 0)
    
    # Add features to dataframe
    for feature, values in feature_values.items():
        df_cyto[feature] = values
    
    return df_cyto

# Process both train and test data
df_cyto = extract_cytogenetic_features(df)
df_eval_cyto = extract_cytogenetic_features(df_eval)

# Merge cytogenetic features with clinical data
X_clinical = X_clinical.merge(df_cyto[['ID'] + list(cyto_features.keys())], on='ID', how='left')

# 1.3 Enhanced mutation features
def extract_mutation_features(clinical_df, mutation_df):
    # Count mutations per patient
    mutation_count = mutation_df.groupby('ID').size().reset_index(name='mutation_count')
    
    # Extract common driver genes in myeloid leukemias
    key_genes = ['TP53', 'FLT3', 'NPM1', 'DNMT3A', 'IDH1', 'IDH2', 'ASXL1', 'RUNX1', 'TET2', 'SRSF2', 'SF3B1']
    
    # Initialize gene mutation status dictionary
    gene_mutations = {gene: [] for gene in key_genes}
    gene_mutations['ID'] = mutation_df['ID'].unique()
    
    # For each patient
    for patient_id in gene_mutations['ID']:
        # Get patient mutations
        patient_muts = mutation_df[mutation_df['ID'] == patient_id]
        
        # Check for mutations in key genes
        for gene in key_genes:
            gene_mutations[gene].append(1 if gene in patient_muts['GENE'].values else 0)
    
    # Convert to DataFrame
    gene_mutations_df = pd.DataFrame(gene_mutations)
    
    # Calculate VAF statistics for each patient
    vaf_stats = mutation_df.groupby('ID').agg(
        mean_vaf=('VAF', 'mean'),
        max_vaf=('VAF', 'max'),
        min_vaf=('VAF', 'min'),
        std_vaf=('VAF', 'std')
    ).reset_index()
    
    # Fill NaN in std_vaf (patients with only 1 mutation)
    vaf_stats['std_vaf'] = vaf_stats['std_vaf'].fillna(0)
    
    # Count high-impact mutations
    high_impact = mutation_df[mutation_df['EFFECT'].isin(['MODERATE', 'HIGH'])].groupby('ID').size().reset_index(name='high_impact_count')
    
    # Merge all mutation features
    result = clinical_df.merge(mutation_count, on='ID', how='left')
    result = result.merge(gene_mutations_df, on='ID', how='left')
    result = result.merge(vaf_stats, on='ID', how='left')
    result = result.merge(high_impact, on='ID', how='left')
    
    # Fill NaN values for patients with no mutations
    for gene in key_genes:
        result[gene] = result[gene].fillna(0)
    result['mutation_count'] = result['mutation_count'].fillna(0)
    result['high_impact_count'] = result['high_impact_count'].fillna(0)
    result['mean_vaf'] = result['mean_vaf'].fillna(0)
    result['max_vaf'] = result['max_vaf'].fillna(0)
    result['min_vaf'] = result['min_vaf'].fillna(0)
    result['std_vaf'] = result['std_vaf'].fillna(0)
    
    return result

# Apply mutation feature extraction
X_full = extract_mutation_features(X_clinical, maf_df)
X_eval_full = extract_mutation_features(df_eval_cyto[['ID'] + clinical_features + list(cyto_features.keys())], maf_eval)

# 1.4 Feature interaction terms
X_full['blast_mutation_interaction'] = X_full['BM_BLAST'] * X_full['mutation_count']
X_eval_full['blast_mutation_interaction'] = X_eval_full['BM_BLAST'] * X_eval_full['mutation_count']

# 1.5 Handle categorical variables
X_full = pd.get_dummies(X_full, columns=['CENTER'], drop_first=True)
X_eval_full = pd.get_dummies(X_eval_full, columns=['CENTER'], drop_first=True)

# Ensure both dataframes have the same columns
missing_cols = set(X_full.columns) - set(X_eval_full.columns)
for col in missing_cols:
    X_eval_full[col] = 0
X_eval_full = X_eval_full[X_full.columns]


In [6]:

# ==== 2. MODEL TRAINING WITH RANDOM SURVIVAL FOREST ====

# Save ID column and drop it for modeling
X_ids = X_full['ID']
X_eval_ids = X_eval_full['ID']
X_full = X_full.drop('ID', axis=1)
X_eval_full = X_eval_full.drop('ID', axis=1)

# Create the survival data format
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.25, random_state=42)
X_train, y_train = X_full, y

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_eval_scaled = scaler.transform(X_eval_full)

# Convert back to DataFrame to keep column names
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns)
X_eval = pd.DataFrame(X_eval_scaled, columns=X_eval_full.columns)

print("Training Random Survival Forest model...")

# Define the RSF model with optimized hyperparameters
rsf = RandomSurvivalForest(
    n_estimators=300,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

# Train the model
rsf.fit(X_train, y_train)

# Evaluate the model
rsf_pred_train = rsf.predict(X_train)
rsf_pred_test = rsf.predict(X_test)

rsf_c_index_train = concordance_index_ipcw(y_train, y_train, rsf_pred_train, tau=7)[0]
rsf_c_index_test = concordance_index_ipcw(y_train, y_test, rsf_pred_test, tau=7)[0]

print(f"Random Survival Forest Concordance Index IPCW on train: {rsf_c_index_train:.4f}")
print(f"Random Survival Forest Concordance Index IPCW on test: {rsf_c_index_test:.4f}")


Training Random Survival Forest model...
Random Survival Forest Concordance Index IPCW on train: 0.7900
Random Survival Forest Concordance Index IPCW on test: 0.7857


In [7]:

# ==== 3. MAKE PREDICTIONS FOR SUBMISSION ====

# Generate predictions on the evaluation set
prediction_on_test_set = rsf.predict(X_eval)

# Create submission dataframe
submission_rsf = pd.Series(prediction_on_test_set, index=X_eval_ids, name='risk_score')
submission_rsf.to_csv('./rsf_submission.csv')
